# AIHW Suicide Mortality Data: Exploratory Data Assessment

**Dataset:** AIHW National Mortality Database (NMD): Suicide and Self-harm Monitoring, 2024 release  
**Source:** Australian Institute of Health and Welfare (AIHW)  
**ICD-10 codes covered:** X60–X84, Y87.0 (intentional self-harm)

## Purpose of this notebook

Before any analysis or visualisation, this notebook systematically examines the structure, 
coverage, and data quality of five selected tables from the AIHW NMD release. This step 
is essential because health administrative datasets, including those from AIHW carry 
known limitations: suppressed small counts, varying temporal coverage across tables, 
and wide format structures that require reshaping before analysis.

**Tables assessed:**
- NMD 1: Suicide by sex, 1907–2024 (national long-run trend)
- NMD 5: Suicide by remoteness area and sex, 2001–2024 (geographic equity)
- NMD 13a: Suicide among children and young people, 2001–2024 (youth burden)
- NMD 16: Deaths of despair from suicide, alcohol, drug-related deaths, 1997–2024
- NMD 17: Psychosocial risk factors by sex and age group, 2017–2024

**A note on data sensitivity:** This dataset records deaths by suicide. All findings 
will be presented in accordance with AIHW and Mindframe reporting guidelines,
clinically, factually and without sensationalised language.

## 1. Setup and imports

We use `openpyxl` to load the raw Excel file, and `pandas` for structural inspection. 
We are NOT doing analysis yet, just loading and diagnosing.

In [3]:
import pandas as pd
import openpyxl

# Path to the AIHW file - update this if your file is in a different location
FILE_PATH = 'Data/AIHW_suicide-and-self-harm-monitoring_NMD_suicide_2024.xlsx'

# These are the five tables we are working with
# We'll refer to them by these short names throughout the project
TABLES = {
    'NMD1':  'Table NMD 1',
    'NMD5':  'Table NMD 5',
    'NMD13': 'Table NMD 13a',
    'NMD16': 'Table NMD 16',
    'NMD17': 'Table NMD 17'
}

print('Libraries loaded successfully.')
print(f'Pandas version: {pd.__version__}')

Libraries loaded successfully.
Pandas version: 2.2.3


## 2. Workbook overview: what sheets exist?

The first step with any multi-sheet Excel file is confirming the workbook structure 
before loading individual sheets. This guards against silent errors where a sheet 
name has changed between file versions.

In [4]:
# Load workbook structure only (no data yet) — much faster for large files
wb = openpyxl.load_workbook(FILE_PATH, read_only=True)

print('All sheets in this workbook:')
for i, name in enumerate(wb.sheetnames):
    print(f'  {i+1:>2}. {name}')

print(f'\nTotal sheets: {len(wb.sheetnames)}')

# Confirm our five target sheets actually exist
print('\nConfirming our five target tables exist:')
for key, sheet_name in TABLES.items():
    exists = sheet_name in wb.sheetnames
    status = '✓ Found' if exists else '✗ MISSING'
    print(f'  {status}: {sheet_name}')

All sheets in this workbook:
   1. Contents
   2. Explanatory notes
   3. Table NMD 1
   4. Table NMD 2
   5. Table NMD 3a
   6. Table NMD 3b
   7. Table NMD 4a
   8. Table NMD 4b
   9. Table NMD 5
  10. Table NMD 6
  11. Table NMD 7
  12. Table NMD 8
  13. Table NMD 9a
  14. Table NMD 9b
  15. Table NMD 10
  16. Table NMD 11
  17. Table NMD 12
  18. Table NMD 13a
  19. Table NMD 13b
  20. Table NMD 14
  21. Table NMD 15
  22. Table NMD 16
  23. Table NMD 17

Total sheets: 23

Confirming our five target tables exist:
  ✓ Found: Table NMD 1
  ✓ Found: Table NMD 5
  ✓ Found: Table NMD 13a
  ✓ Found: Table NMD 16
  ✓ Found: Table NMD 17


## 3. Table by table structural assessment

For each table we want to know four things before writing any analysis code:

1. **What does the raw layout look like?** (header rows, shape)
2. **What time period does it actually cover?** (some tables start in 1907, others 2017)
3. **How is missingness represented?** (AIHW uses `'. .'` for not available and `'n.p.'` for not publishable due to small numbers — these are different and matter clinically)
4. **What is the format?** (wide vs long, this determines what cleaning is needed)

### Why the distinction between `'. .'` and `'n.p.'` matters

This is a domain knowledge point worth pausing on. `'. .'` typically means data was 
not collected or not available for that period. `'n.p.'` (not publishable) means data 
exists but has been suppressed because the count is too small (usually fewer than 5 
deaths), to publish without risking re-identification or producing unreliable rates. 
In a generic data science context you might replace both with `NaN` and move on. 
In a health data context, `'n.p.'` values are informative: they tell you something 
real happened in that cell, just not how much. Treating them as equivalent to 
"no data" would be an analytical error.

In [14]:
def inspect_table(file_path, sheet_name, preview_rows=8):
    """
    Loads a sheet and prints a structural summary:
    - Raw shape (rows x columns)
    - First N rows as raw preview
    - Count of each suppression/missing code found
    """
    # header=None means pandas won't try to guess the header row
    # We want to see the raw layout exactly as AIHW structured it
    df = pd.read_excel(file_path, sheet_name=sheet_name, header=None)
    
    print(f'{'='*60}')
    print(f'SHEET: {sheet_name}')
    print(f'{'='*60}')
    print(f'Raw dimensions: {df.shape[0]} rows x {df.shape[1]} columns')
    
    # Count suppression codes — these are strings AIHW uses for missing/suppressed data
    suppression_codes = ['. .', 'n.p.', 'n.a.', 'N/A']
    print('\nSuppression/missing value counts:')
    for code in suppression_codes:
        # Check entire dataframe for each code
        count = (df == code).sum().sum()
        if count > 0:
            print(f'  "{code}"  →  {count} cells')
    
    # Also check for actual NaN values
    nan_count = df.isna().sum().sum()
    print(f'  NaN (blank) →  {nan_count} cells')
    
    print(f'\nFirst {preview_rows} rows (raw layout):')
    # Only show first 6 columns to keep output readable
    print(df.iloc[:preview_rows, :6].to_string())
    print()

In [7]:
# Run the inspection on NMD 1 — national trend by sex
# Expected: wide format with years as columns, running from 1907 to 2024
inspect_table(FILE_PATH, TABLES['NMD1'])

SHEET: Table NMD 1
Raw dimensions: 24 rows x 120 columns

Suppression/missing value counts:
  ". ."  →  354 cells
  NaN (blank) →  1191 cells

First 8 rows (raw layout):
                                           0                                    1          2          3          4          5
0  Table NMD 1: Suicide by sex, 1907 to 2024                                  NaN        NaN        NaN        NaN        NaN
1                                        Sex                              Measure       1907       1908       1909       1910
2                                      Males  Age-standardised rate (per 100,000)  26.694265  25.347353  25.010777  26.719395
3                                    Females  Age-standardised rate (per 100,000)   5.222553   4.666333   5.307562   4.935708
4                                    Persons  Age-standardised rate (per 100,000)  16.897077    15.9156  15.924692  16.638067
5                                      Males             Crude rate (per 1

In [8]:
# NMD 5 — remoteness area by sex
# Expected: wide format, years 2001-2024, rows for each remoteness category
inspect_table(FILE_PATH, TABLES['NMD5'])

SHEET: Table NMD 5
Raw dimensions: 67 rows x 27 columns

Suppression/missing value counts:
  "n.p."  →  50 cells
  NaN (blank) →  313 cells

First 8 rows (raw layout):
                                                               0        1                                    2          3          4          5
0  Table NMD 5: Suicide by Remoteness area and sex, 2001 to 2024      NaN                                  NaN        NaN        NaN        NaN
1                                                Remoteness area      Sex                              Measure       2001       2002       2003
2                                                   Major Cities  Males    Age-standardised rate (per 100,000)       18.6       16.7       16.4
3                                                 Inner Regional  Males    Age-standardised rate (per 100,000)       22.4       21.6         20
4                                                 Outer Regional  Males    Age-standardised rate (per 100,000)  

In [9]:
# NMD 13a — children and young people
# Expected: wide format, years 2001-2024, rows for different age bands
inspect_table(FILE_PATH, TABLES['NMD13'])

SHEET: Table NMD 13a
Raw dimensions: 30 rows x 26 columns

Suppression/missing value counts:
  "n.p."  →  2 cells
  NaN (blank) →  201 cells

First 8 rows (raw layout):
                                                                      0                                1       2       3       4       5
0  Table NMD 13a: Suicide among children and young people, 2001 to 2024                              NaN     NaN     NaN     NaN     NaN
1                                                             Age group                          Measure  2001.0  2002.0  2003.0  2004.0
2                                                                  0–14  Age-specific rate (per 100,000)     0.3     0.2     0.3     0.2
3                                                                  5–14  Age-specific rate (per 100,000)     0.4     0.3     0.5     0.3
4                                                                  5–17  Age-specific rate (per 100,000)     1.7     1.7     1.7     1.0
5        

In [10]:
# NMD 16 — deaths of despair
# Expected: wide format, three causes of death, 1997-2024
inspect_table(FILE_PATH, TABLES['NMD16'])

SHEET: Table NMD 16
Raw dimensions: 48 rows x 31 columns

Suppression/missing value counts:
  NaN (blank) →  331 cells

First 8 rows (raw layout):
                                                                                                      0        1                                    2            3            4            5
0  Table NMD 16: Deaths of despair: Suicide, alcohol and other drug-related deaths by sex, 1997 to 2024      NaN                                  NaN          NaN          NaN          NaN
1                                                                              Selected causes of death      Sex                              Measure  1997.000000  1998.000000  1999.000000
2                                                                 Alcoholic liver disease and cirrhosis    Males  Age-standardised rate (per 100,000)     9.052593     8.309065     7.929171
3                                                                                               S

In [11]:
# NMD 17 — psychosocial risk factors
# This one is DIFFERENT — already long format, uses ICD-10 Z-codes as risk factor labels
# ICD-10 Z-codes cover 'factors influencing health status and contact with health services'
# Z915 = personal history of self-harm, Z635 = disruption of family by separation/divorce, etc.
inspect_table(FILE_PATH, TABLES['NMD17'])

SHEET: Table NMD 17
Raw dimensions: 1352 rows x 7 columns

Suppression/missing value counts:
  NaN (blank) →  43 cells

First 8 rows (raw layout):
                                                                                                       0               1     2         3          4       5
0  Table NMD 17: Suicide and most frequent psychosocial risk factors, by sex and age group, 2017 to 2024             NaN   NaN       NaN        NaN     NaN
1                                                                                           Risk factor   Frequency rank  Year       Sex  Age group  Number
2                                                                     Z915 Personal history of self-harm               1  2017  Females        0–24      53
3                                                    Z635 Disruption of family by separation and divorce               2  2017  Females        0–24      17
4                                                   Z630 Problems in rela

## 4. Temporal coverage comparison

One of the first planning decisions in any multi-table analysis is understanding 
what time periods overlap. You cannot combine or compare tables across periods 
where one has data and another does not.

This cell builds a simple coverage summary so we can visualise the overlap 
or lack of it, before writing any analysis.

In [12]:
# Temporal coverage summary — built from what we know about each table
# (confirmed by inspection above)
coverage = {
    'NMD 1  — National trend (sex)':          (1907, 2024),
    'NMD 5  — Remoteness area':               (2001, 2024),
    'NMD 13a — Children & young people':       (2001, 2024),
    'NMD 16 — Deaths of despair':             (1997, 2024),
    'NMD 17 — Psychosocial risk factors':     (2017, 2024),
}

print('Temporal coverage by table:')
print(f'{"Table":<45} {"Start":>6} {"End":>6} {"Years":>6}')
print('-' * 65)
for table, (start, end) in coverage.items():
    years = end - start + 1
    print(f'{table:<45} {start:>6} {end:>6} {years:>6}')

print()
print('Key implication for analysis:')
print('  - NMD 17 is the shortest series (2017-2024, only 8 years).')
print('  - Trend analysis on NMD 17 will be limited — interpret patterns cautiously.')
print('  - For cross-table comparisons, the common period is 2017-2024.')
print('  - The long run trend analysis (NMD 1) should focus on age-standardised rates,')
print('    not crude rates, which are unavailable before 1968.')

Temporal coverage by table:
Table                                          Start    End  Years
-----------------------------------------------------------------
NMD 1  — National trend (sex)                   1907   2024    118
NMD 5  — Remoteness area                        2001   2024     24
NMD 13a — Children & young people               2001   2024     24
NMD 16 — Deaths of despair                      1997   2024     28
NMD 17 — Psychosocial risk factors              2017   2024      8

Key implication for analysis:
  - NMD 17 is the shortest series (2017-2024, only 8 years).
  - Trend analysis on NMD 17 will be limited — interpret patterns cautiously.
  - For cross-table comparisons, the common period is 2017-2024.
  - The long run trend analysis (NMD 1) should focus on age-standardised rates,
    not crude rates, which are unavailable before 1968.


## 5. Format diagnosis: wide vs long

Understanding whether each table is in wide or long format is critical because 
it determines what cleaning operation is needed. This is not just a technical 
detail, it reflects how AIHW structured the data for human reading (wide) 
versus how pandas and matplotlib expect data for programmatic analysis (long).

**Wide format** means each year is its own column. Easy to read as a spreadsheet, 
hard to plot directly. Requires `pd.melt()` to convert to long.

**Long format** means each row is one observation (one combination of year, 
sex, age group, measure). This is what we want for analysis.

In [13]:
# Format summary based on inspection
format_summary = [
    ('NMD 1',  'Wide',  'Years as columns (1907-2024)',  'pd.melt() after header cleanup'),
    ('NMD 5',  'Wide',  'Years as columns (2001-2024)',  'pd.melt() after header cleanup'),
    ('NMD 13', 'Wide',  'Years as columns (2001-2024)',  'pd.melt() after header cleanup'),
    ('NMD 16', 'Wide',  'Years as columns (1997-2024)',  'pd.melt() after header cleanup'),
    ('NMD 17', 'Long',  'One row per risk factor/year/sex/age', 'Header rename only — already analysis-ready'),
]

print(f'{"Table":<10} {"Format":<6} {"Structure":<40} {"Cleaning needed"}')
print('-' * 95)
for row in format_summary:
    print(f'{row[0]:<10} {row[1]:<6} {row[2]:<40} {row[3]}')

print()
print('Note: NMD 17 uses ICD-10 Z-codes as risk factor identifiers (e.g. Z915 = personal')
print('history of self-harm). These will be decoded into plain-English labels in the')
print('cleaning notebook to make the visualisations readable.')

Table      Format Structure                                Cleaning needed
-----------------------------------------------------------------------------------------------
NMD 1      Wide   Years as columns (1907-2024)             pd.melt() after header cleanup
NMD 5      Wide   Years as columns (2001-2024)             pd.melt() after header cleanup
NMD 13     Wide   Years as columns (2001-2024)             pd.melt() after header cleanup
NMD 16     Wide   Years as columns (1997-2024)             pd.melt() after header cleanup
NMD 17     Long   One row per risk factor/year/sex/age     Header rename only — already analysis-ready

Note: NMD 17 uses ICD-10 Z-codes as risk factor identifiers (e.g. Z915 = personal
history of self-harm). These will be decoded into plain-English labels in the
cleaning notebook to make the visualisations readable.


## 6. Summary: what we know before writing a single line of analysis

This section records the key findings from the exploration phase. In a professional 
setting, this would go into a data quality report before the analysis is run.

**What we have:**
- Five tables with complementary perspectives: national trend, geographic equity, 
  youth burden, deaths of despair and psychosocial risk factors.
- Clean, authoritative government data published by AIHW in April 2026.
- Age-standardised rates available in NMD 1 and NMD 5, enabling valid comparisons 
  across time and geography.

**Known limitations to carry forward into analysis:**
- **Coronial lag:** Recent years (2022–2024) may undercount suicide deaths where 
  coronial findings are still pending. AIHW notes this in their explanatory notes. 
  Treat the most recent two years as provisional.
- **Suppressed cells (n.p.):** Small counts in remote areas and young age groups 
  are suppressed. These represent real events, they are not zero values.
- **NMD 17 coverage:** Only 2017–2024. Limited for trend analysis; useful for 
  cross-sectional comparison of risk factor profiles by age and sex.
- **ICD-10 Z-codes in NMD 17:** Risk factors are coded using Z-codes. Interpreting 
  these correctly requires clinical literacy, a CS graduate without health training 
  might not distinguish between Z915 (history of self-harm) and Z630 
  (relationship problems with spouse/partner).

**Next step:** 02_data_cleaning.ipynb reshapes each table from wide to long format, 
handle suppression codes appropriately and produce clean dataframes ready for analysis.